# Hospital Length of Stay Prediction

**Tabular Regression Project** — Predict hospital admission length of stay (days) from patient and clinical features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `length_of_stay_days`

## 2 · Project Overview

We predict the **length of hospital stay** (in days) from patient demographics, clinical indicators, and admission details. This is a critical healthcare operations problem — accurate LOS predictions enable better bed management, staffing, and discharge planning.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given a patient's age, BMI, number of diagnoses, procedures performed, admission type, department, ICU status, previous admissions, and insurance type, predict the **number of days** they will stay in the hospital.

## 5 · Why This Project Matters

- **Hospital capacity management** depends on accurate LOS forecasting.
- Early prediction enables proactive **discharge planning** and reduces bottlenecks.
- **Cost estimation** for patients and insurers requires LOS prediction.
- Identifying high-LOS patients allows **targeted interventions** to reduce stays.
- Critical for pandemic preparedness and surge capacity planning.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,500 |
| **Features** | 10 (age, gender, BMI, diagnoses, admission type, department, procedures, ICU, previous admissions, insurance) |
| **Target** | `length_of_stay_days` (integer, days) |
| **Categorical** | admission_type (3), department (6), insurance_type (4) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "length_of_stay_days"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8500
age = np.random.randint(18, 95, n)
gender = np.random.choice([0, 1], n)
bmi = np.random.normal(27, 6, n).clip(15, 55)
num_diagnoses = np.random.randint(1, 12, n)
admission_type = np.random.choice(["emergency", "urgent", "elective"], n, p=[0.4, 0.3, 0.3])
department = np.random.choice(["general", "cardiology", "orthopedics", "neurology",
                                "oncology", "pulmonology"], n)
num_procedures = np.random.randint(0, 8, n)
icu_flag = np.random.choice([0, 1], n, p=[0.8, 0.2])
prev_admissions = np.random.randint(0, 10, n)
insurance = np.random.choice(["private", "medicare", "medicaid", "self_pay"], n)

adm_factor = np.where(admission_type == "emergency", 3,
             np.where(admission_type == "urgent", 2, 0)).astype(float)

los = (2 + age * 0.05 + num_diagnoses * 0.8
       + num_procedures * 1.2 + icu_flag * 5
       + adm_factor + prev_admissions * 0.3
       + bmi * 0.03 + np.random.exponential(1.5, n))
los = np.maximum(los, 1).astype(int)

df = pd.DataFrame({
    "age": age, "gender": gender, "bmi": bmi,
    "num_diagnoses": num_diagnoses, "admission_type": admission_type,
    "department": department, "num_procedures": num_procedures,
    "icu_admission": icu_flag, "previous_admissions": prev_admissions,
    "insurance_type": insurance,
    "length_of_stay_days": los
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["age", "num_diagnoses", "num_procedures",
                          "bmi", "previous_admissions", "icu_admission"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["length_of_stay_days"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("LOS (days)"); ax.set_title(f"LOS vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("admission_type")["length_of_stay_days"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Mean LOS by Admission Type"); axes[0].set_xlabel("Days")
df.groupby("department")["length_of_stay_days"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Mean LOS by Department"); axes[1].set_xlabel("Days")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `length_of_stay_days`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("Days")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():.1f} days, Median: {df[TARGET].median():.1f} days")
print(f"Std: {df[TARGET].std():.1f} days, Range: {df[TARGET].min()} - {df[TARGET].max()} days")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["clinical_complexity"] = X_train["num_diagnoses"] + X_train["num_procedures"] * 2 + X_train["icu_admission"] * 5
X_test["clinical_complexity"] = X_test["num_diagnoses"] + X_test["num_procedures"] * 2 + X_test["icu_admission"] * 5

X_train["age_bmi_interaction"] = X_train["age"] * X_train["bmi"] / 100
X_test["age_bmi_interaction"] = X_test["age"] * X_test["bmi"] / 100

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Number of procedures** and **ICU admission** are the strongest LOS predictors.
- **Number of diagnoses** (comorbidity burden) significantly extends stays.
- **Emergency admissions** average 3+ more days than elective.
- **Age** has a moderate positive effect — older patients stay longer.
- **Previous admissions** (readmission risk) modestly increases LOS.

**Business takeaway:** Focus discharge planning resources on patients with high clinical complexity scores. ICU patients and those with multiple procedures drive the bulk of extended stays.

## 26 · Limitations

1. Synthetic data — real LOS depends on specific diagnoses, severity scores (APACHE), and complications.
2. No diagnosis codes (ICD-10) — 'num_diagnoses' is a coarse proxy.
3. No temporal trends — LOS has decreased over decades due to DRG-based payment.
4. Right-skewed target — some patients stay weeks/months (long-tail distribution).
5. No readmission outcome — LOS reduction shouldn't increase readmissions.

## 27 · How to Improve This Project

1. Use real hospital discharge data (MIMIC-IV, HCUP).
2. Add diagnosis codes and severity scores.
3. Include lab values (albumin, hemoglobin) as clinical predictors.
4. Model as count data (Poisson/Negative Binomial) for integer LOS.
5. Add complications-during-stay as time-varying features.

## 28 · Production Considerations

- Deploy at admission for real-time bed management.
- Update predictions daily as clinical status changes.
- Integrate with hospital information systems (HIS/EHR).
- Monitor for prediction drift across departments.

## 29 · Common Mistakes

1. Not treating LOS as count/right-skewed data — standard OLS assumptions may fail.
2. Using discharge information (which is only known at the end) to predict LOS at admission.
3. Ignoring department-specific LOS patterns.
4. Training on all departments together when patterns differ significantly.
5. Not validating that shorter predicted LOS doesn't increase readmission risk.

## 30 · Mini Challenge / Exercises

1. Remove `icu_admission` — how much does RMSE change?
2. Try log(length_of_stay_days) as the target — does it improve predictions?
3. Train separate models for emergency vs. elective — compare R².
4. Add a `diagnoses × procedures` interaction — does it help?
5. Build a model using only admission-time features (no procedures) — how much accuracy is lost?

## 31 · Final Summary / Key Takeaways

1. **Clinical complexity** (procedures + ICU + diagnoses) is the top predictor.
2. **Emergency admissions** are associated with significantly longer stays.
3. **Age and BMI** have moderate positive effects on LOS.
4. **Gradient boosting** handles the skewed, integer-valued target well.
5. **Feature engineering** (clinical complexity score) adds interpretable value.
6. **Production deployment** needs real-time EHR integration and daily prediction updates.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))